# Notebook14 preparation definitions
Purpose: training-only initial concentration calibration, verified original data, and discarded timing. Inputs: pinned Food101 train/development metadata and tagged exports. Outputs: calibration JSON and timing checkpoints. Expected runtime: under five minutes. Checkpoints: checkpoints/<run_id>/calibration/. Resume through unchanged Notebook14 launch cell; immutable evidence cannot be overwritten.

In [ ]:
"""Training-only entropy calibration and discarded full-update timing for Notebook14."""
from pathlib import Path
import copy,gc,hashlib,json,math,random,statistics,time
import torch
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json,digest_file
from fmm_jepa.food101_training import isolated_rng,validate_training_manifest
import types
VARIANTS=('vit','normalized_kernel','compensated_kernel','compensated_wide_kernel')

_MODULES={}
def load(root,name,folder='14_routing_capacity'):
    path=Path(root)/'experiments'/folder/(name+'.ipynb')
    key=str(path.resolve())
    if key in _MODULES:return _MODULES[key]
    module=types.ModuleType('fmm14_'+folder+'_'+name);module.__file__=str(path)
    TaggedNotebookLoader(path).exec_module(module);_MODULES[key]=module;return module

def canonical(x):return json.loads(json.dumps(x,sort_keys=True,allow_nan=False))
def digest(x):return hashlib.sha256(json.dumps(canonical(x),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def immutable(path,value):
    path=Path(path);value=canonical(value)
    if path.exists():
        if json.loads(path.read_text())!=value:raise ValueError('Immutable evidence changed: '+str(path))
    else:atomic_json(path,value)
    return value

def base_config(root):
    cfg=load(root,'preparation','13_product_geometry').base_config(root)
    cfg.pop('kernel13',None)
    cfg.update(numerical_backend='exact-dense-routing-capacity-v1',device='cuda:0')
    cfg['kernel14']={'variant':'vit','depth':4,'heads':3,'semantic_dim':6,'coordinate_dim':8,
       'bandwidths':[.15,.35,.75],'content_scale':.5,'content_length_factor':1.,'checkpoint':False}
    cfg['training'].update(updates=600,checkpoint_every=50,microbatch=128,accumulation_steps=1)
    cfg['evaluation'].update(heldout_enabled=False,batch_size=64,diagnostic_steps=[600])
    cfg['budget'].update(max_seconds=7200,target_seconds=7200,evaluation_reserve_seconds=900)
    cfg['comparison']='Four-pair exploratory full-JEPA screening: source-norm compensation and wider semantic routing; fixed20epoch final probes. No FMM approximation.'
    cfg['resolution_rationale']='128px patch8 width192 depth4 heads3, shared stem/MLPs/norms/predictor/value maps; exact dense interactions.'
    cfg['input_pipeline']={'workers':6};cfg['execution_optimization']=True
    return cfg

def arm_config(base,variant,wide_factor=None):
    if variant not in VARIANTS:raise ValueError('Unknown registered14variant')
    cfg=copy.deepcopy(base);dim=32 if variant=='compensated_wide_kernel'else 6
    cfg['kernel14'].update(variant=variant,semantic_dim=dim,coordinate_dim=dim+2,
      content_length_factor=float(wide_factor if wide_factor is not None else (32/6)**.25)if dim==32 else 1.)
    return cfg

def prepare_bundle(root,artifact,config,verify_images=True):
    root,artifact=Path(root).resolve(),Path(artifact).resolve()
    module=load(root,'preparation','05_scaling')
    def paths(source_root,folder,stage):
        source_root,folder=Path(source_root).resolve(),Path(folder).resolve()
        relative=folder.relative_to(source_root/'artifacts/notebook_14');run_id=relative.parts[0]
        return folder,source_root/'checkpoints'/run_id/'stages'/stage['id']/'calibration',run_id
    module._stage_paths=paths
    return module.prepare_stage_bundle(root,artifact,{'id':'learning','image_size':128,'pretrain_pool':'selected'},config,verify_images=verify_images)

def entropy_calibration(root,artifact,bundle):
    """Match initial wide/compensated6 mixed-row entropy on four TRAIN images; never dev/test."""
    root,artifact=Path(root),Path(artifact);out=artifact/'entropy_calibration.json'
    runner=load(root,'runner');backend=load(root,'encoder')
    cfg=base_config(root);cfg['seeds']['model_init']=20269914
    cfg['model']['feature_seed']=20270914
    ids=random.Random(20269914).sample(list(bundle['train_ids']),4);identity={'source_hashes':runner.stage_definition_hashes(root),'seed':20269914,
      'sample_ids':ids,'purpose':'Unlabeled train-only initial mixed-kernel entropy; calibration models discarded',
      'rule':'First-block mean row entropy match wide32 to compensated6 by fixed scalar content length; fixed spatial bandwidth; log-bisection24 iterations',
      'bounds':[.25,8.],'default_quarter_power_factor':(32/6)**.25}
    if out.exists():
        record=json.loads(out.read_text())
        if record['identity']!=canonical(identity):raise ValueError('Entropy calibration identity changed')
        return record
    from fmm_jepa.food101_data import EvaluationDataset
    dataset=EvaluationDataset(bundle['dataset'],ids,cfg)
    images=torch.stack([dataset[i]['image']for i in range(4)])
    models={}
    for variant in ('compensated_kernel','compensated_wide_kernel'):
        config=arm_config(cfg,variant)
        current,info=runner.initialize_models(config,artifact/'entropy_initialization'/variant)
        models[variant]=current['semantic'].context.eval();del current
    def entropy(encoder,factor=None):
        if factor is not None:encoder.blocks[0].mixer.content_length_factor=float(factor)
        with torch.no_grad():
            x,indices,valid,grid=encoder.embed(images)
            weights=encoder.blocks[0].mixer.kernel_state(encoder.blocks[0].norm1(x),indices,valid,grid)['weights']
            return float(-(weights*weights.clamp_min(1e-30).log()).sum(-1).mean())
    target=entropy(models['compensated_kernel']);wide=models['compensated_wide_kernel']
    default=entropy(wide,(32/6)**.25);low,high=.25,8.;elo,ehi=entropy(wide,low),entropy(wide,high)
    if not elo<=target<=ehi:raise ValueError('Training entropy target outside fixed calibration bracket')
    trace=[]
    for iteration in range(24):
        mid=math.sqrt(low*high);value=entropy(wide,mid);trace.append({'factor':mid,'entropy':value})
        if value<target:low=mid
        else:high=mid
    factor=math.sqrt(low*high);matched=entropy(wide,factor)
    if abs(matched-target)>.002:raise AssertionError('Initial routing concentration failed to match')
    record={'status':'PASS','identity':canonical(identity),'wide_content_length_factor':factor,
      'compensated6_entropy_nats':target,'wide_default_entropy_nats':default,'wide_matched_entropy_nats':matched,
      'absolute_error_nats':abs(matched-target),'trace':trace,'weights_discarded':True,'quality_evidence':False,
      'calibration_scope':'First encoder block, four training images, independent discarded seed; equal initial mean entropy does not imply equal operators.'}
    immutable(out,record);del models,wide;gc.collect();return record

def run_calibration(root,run_id,device):
    root=Path(root);artifact=root/'artifacts/notebook_14'/run_id
    binding=json.loads((artifact/'study_binding.json').read_text());deadline=binding['budget_clock']['deadline_unix']
    wide=float(binding['entropy_calibration']['wide_content_length_factor'])
    shard=artifact/'calibration'/device.replace(':','-');shard.mkdir(parents=True,exist_ok=True)
    runner=load(root,'runner');base=base_config(root);base['device']=device
    base['seeds']['model_init']=20269914;base['model']['feature_seed']=20270914
    bundle=prepare_bundle(root,shard/'data',base,verify_images=True)
    rows=[];trace_reference=None
    for variant in VARIANTS:
        cfg=arm_config(base,variant,wide);case=shard/variant;case.mkdir(parents=True,exist_ok=True)
        checkpoint=root/'checkpoints'/run_id/'calibration'/device.replace(':','-')/variant
        cfg['training']['checkpoint_every']=4
        models,info=runner.initialize_models(cfg,case);kind='vit'if variant=='vit'else'semantic';model=models[kind];del models
        plan={'run_id':run_id,'stage_id':'calibration-'+device.replace(':','-'),'updates':600,'milestones':[600],
          'resume_notebook':'notebooks/14_Routing_Capacity_Screen.ipynb','resume_cell':'launch-routing-screen'}
        protocol={'source_hashes':runner.stage_definition_hashes(root),'purpose':'Eight discarded actual updates; no quality selection','wide_content_length_factor':wide}
        started=time.monotonic()
        result=runner._run_stage_model(plan,kind,cfg,bundle,case,checkpoint,protocol,model=model,stop_after=8,deadline_unix=deadline-15)
        if result['resume']['final_step']!=8 or any(not t['successful']for t in result['traces']):raise RuntimeError('Eight clean timing updates required')
        signature=digest([{k:t[k]for k in('global_step','sample_ids_hash','augmentation_hash','mask_hash')if k in t}for t in result['traces']])
        if trace_reference is not None and signature!=trace_reference:raise AssertionError('Timing examples/masks/augmentations differ')
        trace_reference=signature;measured=result['metrics'][3:8]
        peak=result['performance']['peak_cuda_allocated_bytes']
        if peak>70*1024**3:raise RuntimeError('Timing arm exceeds70GiB memory')
        row={'variant':variant,'device':device,'median_step_seconds':statistics.median(m['step_seconds']for m in measured),
          'peak_bytes':peak,'wall_seconds':time.monotonic()-started,'raw_step_seconds':[m['step_seconds']for m in measured],
          'round_overhead_seconds':max(0.,time.monotonic()-started-sum(m['step_seconds']for m in result['metrics'])),
          'checkpoint_directory':str(checkpoint),'trace_sha256':signature,'status':'PASS'}
        rows.append(row);atomic_json(shard/'progress.json',{'rows':rows,'status':'RUNNING'});print('TIMING',row,flush=True)
        del model,result;gc.collect();torch.cuda.empty_cache()
    report={'status':'PASS','rows':rows,'weights_discarded':True,'quality_evidence':False,'source_hashes':runner.stage_definition_hashes(root)}
    immutable(shard/'result.json',report);return report
